# 01 Preprocessing Pipeline
This notebook documents every preprocessing step applied to the raw dataset, explains the rationale for each decision, and produces the `dataset/processed/` directory used by all 84 training runs.

In [ ]:
import os
import pathlib
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator

RANDOM_SEED   = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR       = pathlib.Path('../dataset/raw')
PROCESSED_DIR = pathlib.Path('../dataset/processed')
CLASSES       = ['cancer', 'no_cancer']
INPUT_SHAPE   = (256, 256)

---
## Step 1: Dataset Provenance

**Source:** STRAMPN Histopathological Images for Ovarian Cancer Prediction  

Total images: **987**  
Task: Binary classification : *cancer* vs *no_cancer*

In [ ]:
# Verify that the raw data directory has the expected structure
for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    n = len(list(cls_dir.glob('*.*'))) if cls_dir.exists() else 0
    print(f"  {cls:12s}: {n:4d} images  (path: {cls_dir})")

---
## Step 2: Directory Setup and Raw Data Inventory

We verify file counts per class and check for corrupt or non-image files before any split is performed.  
The processed directory tree is created if it does not already exist.

In [ ]:
# Create processed directory tree
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        (PROCESSED_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print("Processed directory structure created:")
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        p = PROCESSED_DIR / split / cls
        print(f"  {p}")

In [ ]:
# Inventory raw files — collect valid image paths per class
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
raw_files = {}
for cls in CLASSES:
    files = [
        f for f in sorted((RAW_DIR / cls).glob('*.*'))
        if f.suffix.lower() in VALID_EXTS
    ]
    raw_files[cls] = files
    print(f"  {cls:12s}: {len(files)} valid image files")

---
## Step 3: Train / Validation / Test Split

**Strategy:** Stratified split; performed independently per class to preserve class ratios.  
**Ratios:**  
- First split: 75% train+val / 25% test  
- Second split on train+val: 70% train / 30% val  

This yields approximately:  
- **Train: 518** images  
- **Val:   222** images  
- **Test:  247** images  

**Critical:** `RANDOM_SEED = 42` is fixed globally. This split is performed **once** and the files are copied to `dataset/processed/`. All 84 training runs read from this fixed split, data is never 're-split'.

In [ ]:
import math

def split_files(files, train_ratio=0.525, val_ratio=0.225, seed=42):
    """Stratified split: 52.5% train, 22.5% val, 25% test (≈ 75/25 then 70/30)."""
    rng = random.Random(seed)
    shuffled = files[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = math.floor(n * train_ratio)
    n_val   = math.floor(n * val_ratio)
    return (
        shuffled[:n_train],
        shuffled[n_train:n_train + n_val],
        shuffled[n_train + n_val:],
    )

split_counts = {'train': 0, 'val': 0, 'test': 0}

for cls in CLASSES:
    train_f, val_f, test_f = split_files(raw_files[cls], seed=RANDOM_SEED)
    for files, split in [(train_f, 'train'), (val_f, 'val'), (test_f, 'test')]:
        dest_dir = PROCESSED_DIR / split / cls
        for fp in files:
            shutil.copy2(fp, dest_dir / fp.name)
        split_counts[split] += len(files)
        print(f"  {cls:12s} → {split:5s}: {len(files):4d} images")

print(f"\nTotals → train: {split_counts['train']},  val: {split_counts['val']},  test: {split_counts['test']}")

---
## Step 4: Normalization

All images are normalized using **samplewise standard normalization**:  
- Each image is zero-centered (`samplewise_center=True`)  
- Each image is divided by its own standard deviation (`samplewise_std_normalization=True`)  

Applied uniformly to **train, val, and test** generators.  

---
## Step 5: Augmentation

As specified, data augmentation is applied **only to the training generator** to artificially expand the effective training set size and reduce overfitting. The validation and test generators use normalization only.

**Augmentation parameters:**

| Parameter | Value | Rationale |
|---|---|---|
| `rotation_range` | 30 | Histology slides have no canonical orientation |
| `width_shift_range` | 0.15 | Simulate slight translation variation |
| `height_shift_range` | 0.15 | Simulate slight translation variation |
| `shear_range` | 0.3 | Simulate microscope stage angle variation |
| `zoom_range` | 0.30 | Simulate magnification variation |
| `horizontal_flip` | True | Tissue has no left/right distinction |
| `fill_mode` | `'nearest'` | Fill border pixels without introducing artifacts |

---
## Step 6: Data Generator Construction

Two `ImageDataGenerator` instances are constructed:  
1. **`train_datagen`** — normalization + full augmentation, used for `train/`  
2. **`eval_datagen`** — normalization only, used for `val/` and `test/`

In [ ]:
BATCH_SIZE = 32
IMG_SIZE   = (256, 256)

# Augmented generator (training only) 
train_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.3,
    zoom_range=0.30,
    horizontal_flip=True,
    fill_mode='nearest',
)

# Normalization-only generator (val and test) 
eval_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
)

# Instantiate flow_from_directory generators 
train_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=RANDOM_SEED,
)

val_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

test_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

print(f"Train batches : {len(train_generator)}  ({train_generator.samples} images)")
print(f"Val batches   : {len(val_generator)}    ({val_generator.samples} images)")
print(f"Test batches  : {len(test_generator)}   ({test_generator.samples} images)")
print(f"Class indices : {train_generator.class_indices}")

---
## Step 7: Verification

Display a batch of augmented training images to visually confirm that the augmentation pipeline is functioning correctly. Compare against the corresponding original images.

In [ ]:
# Fetch one batch from the training generator and display the first 8 images
images, labels = next(train_generator)
class_idx_inv = {v: k for k, v in train_generator.class_indices.items()}

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle('Augmented Training Batch (first 8 samples, rows = 2 channels views)', fontweight='bold')

for col_idx in range(min(8, len(images))):
    img = images[col_idx]
    # Rescale for display: shift to [0,1]
    img_disp = (img - img.min()) / (img.max() - img.min() + 1e-8)
    label = class_idx_inv[int(labels[col_idx])]
    axes[0, col_idx].imshow(img_disp)
    axes[0, col_idx].set_title(label, fontsize=8)
    axes[0, col_idx].axis('off')
    # Show single channel (R) to inspect augmentation artifacts
    axes[1, col_idx].imshow(img_disp[:, :, 0], cmap='gray')
    axes[1, col_idx].axis('off')

axes[0, 0].set_ylabel('RGB', fontsize=9)
axes[1, 0].set_ylabel('R channel', fontsize=9)
plt.tight_layout()
plt.show()
print("Augmentation verified. All 7 augmentation transforms are active on the training generator.")